# Final Pipeline: YOLOv8l + SAHI + Tuned NMS
**UCU AI 2026 — Multiple Object Video Detection (HW2)**

**Base:** YOLOv8l epoch-1 `best.pt` (val mAP50 = 0.3835, public LB = 0.23619)

**Pipeline (stacked improvements):**
1. Inference at `imgsz=1280` (vs 640 baseline) for 2× larger object pixels
2. Confidence threshold 0.10 (vs 0.20) to recover faint pedestrians
3. IoU-NMS threshold 0.55 (vs 0.45) to keep adjacent objects in dense scenes
4. SAHI slicing: 640×640 tiles with 20% overlap, merged with full-image predictions
5. Fallback: if SAHI fails or times out, submit the best non-SAHI variant

**Expected outcome:**
- Validation mAP50: 0.42–0.47 (vs 0.3835 baseline)
- Public LB: 0.30–0.34 (vs 0.23619 baseline)

In [1]:
# =============================================================================
# Cell 1 (KAGGLE VERSION): Setup — paths, GPU, best.pt
# =============================================================================
# Prerequisites in Kaggle notebook:
#   - Internet: ON
#   - Accelerator: GPU T4 x2
#   - Input: suchok/yolo-outputs
#   - Input: multiple-object-video-detection-hw-2-ucu-ai-2026
# =============================================================================

import os
import sys
import json
import random
import shutil
import time
from pathlib import Path

# ---- 1a. Install ----------------------------------------------------------
print("[1/5] Installing dependencies...")
os.system("pip install -q ultralytics sahi pyyaml 2>&1 | tail -1")
print("      ✅ Done")

# ---- 1b. Paths (Kaggle-specific) ------------------------------------------
print("\n[2/5] Setting up paths...")
CONTENT    = Path("/kaggle/working")
YOLO_DIR   = Path("/kaggle/input/datasets/suchokkateryna/yolo-outputs")
COMP_DIR   = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
OUT_DIR    = CONTENT / "artifacts_sahi"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify inputs are attached
assert YOLO_DIR.exists(), f"❌ Attach `suchok/yolo-outputs`. Missing: {YOLO_DIR}"
assert COMP_DIR.exists(), f"❌ Attach the competition data. Missing: {COMP_DIR}"

# ---- 1c. GPU check --------------------------------------------------------
print("\n[3/5] Checking GPU...")
import torch
if torch.cuda.is_available():
    print(f"      ✅ GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("      ⚠️ No GPU — set Accelerator to 'GPU T4 x2' in the right sidebar")
    DEVICE = "cpu"

# ---- 1d. Locate best.pt ---------------------------------------------------
print("\n[4/5] Locating best.pt...")
BEST_PT = YOLO_DIR / "runs/detect/visdrone_yolo/yolov8l_800/weights/best.pt"
assert BEST_PT.exists(), f"❌ best.pt not found at {BEST_PT}"
print(f"      ✅ best.pt: {BEST_PT} ({BEST_PT.stat().st_size/1e6:.1f} MB)")

# ---- 1e. Locate competition files -----------------------------------------
print("\n[5/5] Locating competition data...")
TEST_IMAGES = COMP_DIR / "test" / "images"
TEST_CSV    = COMP_DIR / "test_images.csv"
VAL_IMAGES  = COMP_DIR / "val" / "images"
VAL_LABELS  = COMP_DIR / "val" / "labels"

for p, name in [(TEST_IMAGES, "test/images"),
                (TEST_CSV,   "test_images.csv"),
                (VAL_IMAGES, "val/images"),
                (VAL_LABELS, "val/labels")]:
    assert p.exists(), f"❌ missing {name} at {p}"

n_test = len(list(TEST_IMAGES.rglob("*.jpg")))
n_val  = len(list(VAL_IMAGES.rglob("*.jpg")))
print(f"      ✅ test frames: {n_test:,}")
print(f"      ✅ val frames:  {n_val:,}")

# ---- Working constants used by all later cells ----------------------------
VAL_SUBSET = CONTENT / "val_subset"
SUBSET_YAML = OUT_DIR / "val_subset.yaml"

print("\n" + "="*70)
print("✅ SETUP COMPLETE")
print("="*70)
print(f"  DEVICE:       {DEVICE}")
print(f"  BEST_PT:      {BEST_PT}")
print(f"  COMP_DIR:     {COMP_DIR}")
print(f"  OUT_DIR:      {OUT_DIR}")
print(f"  VAL_SUBSET:   {VAL_SUBSET}")

[1/5] Installing dependencies...
      ✅ Done

[2/5] Setting up paths...

[3/5] Checking GPU...
      ✅ GPU: Tesla T4

[4/5] Locating best.pt...
      ✅ best.pt: /kaggle/input/datasets/suchokkateryna/yolo-outputs/runs/detect/visdrone_yolo/yolov8l_800/weights/best.pt (175.1 MB)

[5/5] Locating competition data...
      ✅ test frames: 3,811
      ✅ val frames:  2,846

✅ SETUP COMPLETE
  DEVICE:       cuda
  BEST_PT:      /kaggle/input/datasets/suchokkateryna/yolo-outputs/runs/detect/visdrone_yolo/yolov8l_800/weights/best.pt
  COMP_DIR:     /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
  OUT_DIR:      /kaggle/working/artifacts_sahi
  VAL_SUBSET:   /kaggle/working/val_subset


In [2]:
# =============================================================================
# Cell 2: Build 300-frame val subset for fast pipeline comparison
# =============================================================================
# Why 300 frames? Full val (2,846) × SAHI (4-9× slower) would take hours.
# 300 frames gives statistical significance for A/B testing each technique.
# Final submission uses full test set regardless.

print("[2/6] Building 300-frame val subset...")

if VAL_SUBSET.exists():
    shutil.rmtree(VAL_SUBSET)
VAL_SUBSET.mkdir(parents=True)

all_imgs = sorted(VAL_IMAGES.rglob("*.jpg"))
random.seed(28)
subset = random.sample(all_imgs, 300)
print(f"      Selected {len(subset)} frames from {len(all_imgs):,}")

for img_path in subset:
    rel = img_path.relative_to(VAL_IMAGES)
    dst_img = VAL_SUBSET / "images" / rel
    dst_img.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(img_path, dst_img)

    clip = img_path.parent.name
    stem = img_path.stem
    src_lbl = VAL_LABELS / clip / f"{stem}.txt"
    if src_lbl.exists():
        dst_lbl = VAL_SUBSET / "labels" / clip / f"{stem}.txt"
        dst_lbl.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(src_lbl, dst_lbl)

# Write YAML
import yaml
SUBSET_YAML = OUT_DIR / "val_subset.yaml"
with open(SUBSET_YAML, "w") as f:
    yaml.dump({
        "path": str(VAL_SUBSET),
        "train": "images",
        "val":   "images",
        "nc":    8,
        "names": {0: "person", 1: "bicycle", 2: "car", 3: "van",
                  4: "truck", 5: "tricycle", 6: "bus", 7: "motor"},
    }, f, default_flow_style=False)
print(f"      ✅ YAML: {SUBSET_YAML}")

[2/6] Building 300-frame val subset...
      Selected 300 frames from 2,846
      ✅ YAML: /kaggle/working/artifacts_sahi/val_subset.yaml


In [3]:
# =============================================================================
# Cell 3: EXPERIMENT A — Baseline (imgsz=640, conf=0.20, iou=0.45)
# =============================================================================
# Reproduces the LB 0.23619 setup on our 300-frame val subset.
# This is the reference point — every other experiment is compared to it.

print("\n" + "="*70)
print("EXPERIMENT A: Baseline (imgsz=640, conf=0.20, iou=0.45)")
print("="*70)

from ultralytics import YOLO
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

model = YOLO(str(BEST_PT))

def val_eval_yolo(model, subset_dir, yaml_path, imgsz, conf, iou,
                  tag, device="cuda"):
    """Run model.val() on the subset and return metrics dict."""
    metrics = model.val(
        data=str(yaml_path),
        split="val",
        imgsz=imgsz,
        conf=conf,
        iou=iou,
        device=device,
        project=str(OUT_DIR),
        name=tag,
        plots=False,
        verbose=False,
    )
    return {
        "tag": tag,
        "imgsz": imgsz,
        "conf": conf,
        "iou": iou,
        "mAP50":     float(metrics.box.map50),
        "mAP50-95":  float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall":    float(metrics.box.mr),
        "per_class": {
            model.names[i]: float(metrics.box.ap50[i])
            for i in range(len(metrics.box.ap50))
        },
    }

results = {}

t0 = time.time()
r = val_eval_yolo(model, VAL_SUBSET, SUBSET_YAML,
                  imgsz=640, conf=0.20, iou=0.45,
                  tag="A_baseline_640", device=DEVICE)
results["A"] = r
print(f"\n  mAP50:     {r['mAP50']:.5f}")
print(f"  mAP50-95:  {r['mAP50-95']:.5f}")
print(f"  Precision: {r['precision']:.5f}")
print(f"  Recall:    {r['recall']:.5f}")
print(f"  Time:      {(time.time()-t0)/60:.1f} min")
print(f"\n  Baseline reference: LB score was 0.23619 at this config")


EXPERIMENT A: Baseline (imgsz=640, conf=0.20, iou=0.45)
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 112 layers, 43,612,776 parameters, 0 gradients, 164.8 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 16.4±9.0 MB/s, size: 286.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/val_subset/labels/uav0000086_00000_v... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 117.6it/s 2.6s
val: New cache created: /kaggle/working/val_subset/labels/uav0000086_00000_v.cache
               

In [4]:
# =============================================================================
# Cell 4: EXPERIMENT B — imgsz=1280, conf=0.10, iou=0.55
# =============================================================================
# Tests three simultaneous improvements:
#   - imgsz 640 -> 1280: 2x resolution for small objects
#   - conf 0.20 -> 0.10: recover faint detections
#   - iou  0.45 -> 0.55: keep adjacent objects in dense scenes

print("\n" + "="*70)
print("EXPERIMENT B: imgsz=1280, conf=0.10, iou=0.55")
print("="*70)

t0 = time.time()
r = val_eval_yolo(model, VAL_SUBSET, SUBSET_YAML,
                  imgsz=1280, conf=0.10, iou=0.55,
                  tag="B_tuned_1280", device=DEVICE)
results["B"] = r
print(f"\n  mAP50:     {r['mAP50']:.5f}  ({r['mAP50'] - results['A']['mAP50']:+.5f} vs A)")
print(f"  mAP50-95:  {r['mAP50-95']:.5f}")
print(f"  Precision: {r['precision']:.5f}")
print(f"  Recall:    {r['recall']:.5f}")
print(f"  Time:      {(time.time()-t0)/60:.1f} min")

# Per-class improvement
print("\n  Per-class mAP50 vs baseline:")
print(f"  {'Class':<12} {'A (640)':>10} {'B (1280)':>10} {'Δ':>10}")
for cls in results["A"]["per_class"]:
    a = results["A"]["per_class"][cls]
    b = results["B"]["per_class"][cls]
    print(f"  {cls:<12} {a:>10.4f} {b:>10.4f} {b-a:>+10.4f}")


EXPERIMENT B: imgsz=1280, conf=0.10, iou=0.55
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 112 layers, 43,612,776 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 272.7±153.9 MB/s, size: 450.5 KB)
val: Scanning /kaggle/working/val_subset/labels/uav0000086_00000_v.cache... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 89.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.3s/it 24.9s
                   all        300      12320      0.529       0.45      0.412      0.205
Speed: 1.7ms preprocess, 76.8ms inference, 0.0ms loss, 0.9ms postprocess per image

  mAP50:     0.41208  (+0.08791 vs A)
  mAP50-95:  0.20495
  Precision: 0.52893
  Recall:    0.45037
  Time:      0.4 min

  Per-class mAP50 vs baseline:
  Class           A (640)   B (1280)          Δ
  person           0.5686     0.6770    +0.

In [5]:
# =============================================================================
# Cell 5: EXPERIMENT C — SAHI slicing on top of B's config
# =============================================================================
# SAHI runs the detector on overlapping 640×640 tiles, then merges.
# This is the single biggest lever for VisDrone small objects.

print("\n" + "="*70)
print("EXPERIMENT C: SAHI slicing (slice=640, overlap=0.2)")
print("="*70)

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import contextlib, io

# Build SAHI model once — reused for val and test
def make_sahi_model(conf_threshold=0.10, device="cuda"):
    return AutoDetectionModel.from_pretrained(
        model_type="yolov8",
        model_path=str(BEST_PT),
        confidence_threshold=conf_threshold,
        device=device,
        image_size=1280,       # match B's imgsz
    )

sahi_model = make_sahi_model(conf_threshold=0.10, device=DEVICE)
print("  SAHI model loaded")

def val_eval_sahi(sahi_model, subset_dir, yaml_path, tag,
                  slice_size=640, overlap=0.2):
    """Run SAHI on the val subset and evaluate with pycocotools."""
    # Load GT
    coco_gt = COCO()
    # Build COCO GT directly from subset
    images_list = []
    anns_list = []
    cats_list = [{"id": i+1, "name": n} for i, n in enumerate(
        ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
    )]

    all_imgs = sorted((subset_dir / "images").rglob("*.jpg"))
    for img_id, img_path in enumerate(all_imgs, start=1):
        import cv2
        im = cv2.imread(str(img_path))
        h, w = im.shape[:2]
        images_list.append({"id": img_id, "file_name": str(img_path),
                            "width": w, "height": h})
        lbl_path = subset_dir / "labels" / img_path.parent.name / f"{img_path.stem}.txt"
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                ww = bw * w
                hh = bh * h
                anns_list.append({
                    "id": len(anns_list)+1,
                    "image_id": img_id,
                    "category_id": cls+1,
                    "bbox": [x1, y1, ww, hh],
                    "area": ww*hh,
                    "iscrowd": 0,
                })

    coco_gt.dataset = {"images": images_list, "annotations": anns_list,
                       "categories": cats_list}
    coco_gt.createIndex()

    # SAHI predictions
    preds = []
    id_map = {str(p): i+1 for i, p in enumerate(all_imgs)}
    for img_path in tqdm(all_imgs, desc=f"  SAHI {tag}"):
        result = get_sliced_prediction(
            str(img_path), sahi_model,
            slice_height=slice_size, slice_width=slice_size,
            overlap_height_ratio=overlap, overlap_width_ratio=overlap,
            postprocess_type="NMS",
            postprocess_match_metric="IOU",
            postprocess_match_threshold=0.55,
            verbose=0,
        )
        for p in result.object_prediction_list:
            bb = p.bbox
            preds.append({
                "image_id": id_map[str(img_path)],
                "category_id": int(p.category.id)+1,
                "bbox": [bb.minx, bb.miny, bb.maxx-bb.minx, bb.maxy-bb.miny],
                "score": float(p.score.value),
            })

    # Evaluate
    if len(preds) == 0:
        return {"tag": tag, "mAP50": 0.0, "mAP50-95": 0.0,
                "precision": 0.0, "recall": 0.0, "per_class": {}}

    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(preds)
        e = COCOeval(coco_gt, coco_dt, "bbox")
        e.evaluate(); e.accumulate(); e.summarize()

    per_class = {}
    for i, cls in enumerate(["person", "bicycle", "car", "van", "truck",
                             "tricycle", "bus", "motor"]):
        if i < len(e.stats):  # AP50 is stats[1]; per-class is in e.eval
            pass  # skip per-class for SAHI (would need extra parsing)

    return {
        "tag": tag,
        "slice": slice_size,
        "overlap": overlap,
        "mAP50":     float(e.stats[1]),
        "mAP50-95":  float(e.stats[0]),
        "precision": float(e.stats[1]),  # placeholder
        "recall":    float(e.stats[8]) if len(e.stats) > 8 else 0.0,
        "per_class": per_class,
    }

t0 = time.time()
r = val_eval_sahi(sahi_model, VAL_SUBSET, SUBSET_YAML,
                  tag="C_sahi_640", slice_size=640, overlap=0.2)
results["C"] = r
print(f"\n  mAP50:     {r['mAP50']:.5f}  ({r['mAP50'] - results['B']['mAP50']:+.5f} vs B)")
print(f"  mAP50-95:  {r['mAP50-95']:.5f}")
print(f"  Time:      {(time.time()-t0)/60:.1f} min")


EXPERIMENT C: SAHI slicing (slice=640, overlap=0.2)
  SAHI model loaded
creating index...
index created!


  SAHI C_sahi_640:   0%|          | 0/300 [00:00<?, ?it/s]


  mAP50:     0.31721  (-0.09487 vs B)
  mAP50-95:  0.14810
  Time:      26.2 min


In [6]:
# =============================================================================
# Cell 6: Summary — all experiments side by side
# =============================================================================
print("\n" + "="*70)
print("EXPERIMENT SUMMARY")
print("="*70)
print(f"{'Exp':<4} {'Config':<35} {'mAP50':>10} {'Δ vs A':>10}")
print("-"*70)
base = results["A"]["mAP50"]
for k, r in results.items():
    if k == "A":
        print(f"{k:<4} {'Baseline (imgsz=640, conf=0.20)':<35} {r['mAP50']:>10.5f} {'—':>10}")
    elif k == "B":
        print(f"{k:<4} {'imgsz=1280, conf=0.10, iou=0.55':<35} {r['mAP50']:>10.5f} {r['mAP50']-base:>+10.5f}")
    elif k == "C":
        print(f"{k:<4} {'SAHI slice=640 (best of B)':<35} {r['mAP50']:>10.5f} {r['mAP50']-base:>+10.5f}")

# Pick winner
winner_key = max(results.keys(), key=lambda k: results[k]["mAP50"])
print(f"\n🏆 Best config: {winner_key} (mAP50 = {results[winner_key]['mAP50']:.5f})")

# Save
with open(OUT_DIR / "experiments.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"✅ Saved: {OUT_DIR / 'experiments.json'}")

# Decide which pipeline to use for test inference
USE_SAHI = results["C"]["mAP50"] > results["B"]["mAP50"]
if USE_SAHI:
    print(f"\n✅ SAHI adds {results['C']['mAP50']-results['B']['mAP50']:+.5f} → using SAHI for test")
else:
    print(f"\n⚠️ SAHI underperforms B → using B (no SAHI) for test")


EXPERIMENT SUMMARY
Exp  Config                                   mAP50     Δ vs A
----------------------------------------------------------------------
A    Baseline (imgsz=640, conf=0.20)        0.32418          —
B    imgsz=1280, conf=0.10, iou=0.55        0.41208   +0.08791
C    SAHI slice=640 (best of B)             0.31721   -0.00696

🏆 Best config: B (mAP50 = 0.41208)
✅ Saved: /kaggle/working/artifacts_sahi/experiments.json

⚠️ SAHI underperforms B → using B (no SAHI) for test


In [7]:
# =============================================================================
# Cell 7: Test-set inference with the winning pipeline
# =============================================================================
print("\n" + "="*70)
print(f"TEST INFERENCE — {'SAHI slice=640' if USE_SAHI else 'B (imgsz=1280, conf=0.10, iou=0.55)'}")
print("="*70)

test_df = pd.read_csv(TEST_CSV)
print(f"  {len(test_df):,} test frames")

CLASS_NAMES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
sub_rows = []
t0 = time.time()

def infer_one(img_path):
    """Return list of (class_id, score, x1, y1, w, h) in ORIGINAL pixels."""
    if USE_SAHI:
        result = get_sliced_prediction(
            str(img_path), sahi_model,
            slice_height=640, slice_width=640,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="NMS",
            postprocess_match_metric="IOU",
            postprocess_match_threshold=0.55,
            verbose=0,
        )
        out = []
        for p in result.object_prediction_list:
            bb = p.bbox
            out.append((int(p.category.id), float(p.score.value),
                        bb.minx, bb.miny, bb.maxx-bb.minx, bb.maxy-bb.miny))
        return out
    else:
        res = model.predict(
            source=str(img_path), imgsz=1280, conf=0.10, iou=0.55,
            max_det=500, device=DEVICE, verbose=False,
        )[0]
        if res.boxes is None or len(res.boxes) == 0:
            return []
        xyxy = res.boxes.xyxy.cpu().numpy()
        cls  = res.boxes.cls.cpu().numpy().astype(int)
        scores = res.boxes.conf.cpu().numpy()
        return [(int(c), float(s), float(x1), float(y1),
                 float(x2-x1), float(y2-y1))
                for (x1,y1,x2,y2),c,s in zip(xyxy,cls,scores)]

for i, (_, row) in enumerate(test_df.iterrows(), start=1):
    img_path = COMP_DIR / row["file_name"]
    if not img_path.exists():
        img_path = Path(row["file_name"])

    try:
        detections = infer_one(img_path)
    except Exception as e:
        print(f"  ⚠️ {row['image_id']}: {e}")
        detections = []

    # Sort by score, cap 500
    detections.sort(key=lambda x: -x[1])
    detections = detections[:500]

    parts = []
    for cls, score, x, y, w, h in detections:
        if w <= 0 or h <= 0:
            continue
        parts.append(f"{cls} {score:.4f} {x:.1f} {y:.1f} {w:.1f} {h:.1f}")

    sub_rows.append({"image_id": row["image_id"],
                     "PredictionString": " ".join(parts)})

    if i % 50 == 0:
        elapsed = time.time() - t0
        spf = elapsed / i
        eta_min = (len(test_df) - i) * spf / 60
        print(f"  [{i}/{len(test_df)}] elapsed={elapsed/60:.1f}min ETA={eta_min:.0f}min")

# Save
sub_df = pd.DataFrame(sub_rows)
sub_path = OUT_DIR / "submission_final.csv"
sub_df.to_csv(sub_path, index=False)
non_empty = (sub_df["PredictionString"].str.len() > 0).sum()
print(f"\n✅ {sub_path}")
print(f"   non-empty: {non_empty}/{len(sub_df)} ({100*non_empty/len(sub_df):.1f}%)")
print(f"   size: {sub_path.stat().st_size/1e6:.1f} MB")
print(f"   total time: {(time.time()-t0)/60:.1f} min")


TEST INFERENCE — B (imgsz=1280, conf=0.10, iou=0.55)
  3,811 test frames
  [50/3811] elapsed=0.2min ETA=12min
  [100/3811] elapsed=0.3min ETA=12min
  [150/3811] elapsed=0.5min ETA=11min
  [200/3811] elapsed=0.6min ETA=11min
  [250/3811] elapsed=0.8min ETA=11min
  [300/3811] elapsed=0.9min ETA=11min
  [350/3811] elapsed=1.1min ETA=10min
  [400/3811] elapsed=1.2min ETA=10min
  [450/3811] elapsed=1.4min ETA=10min
  [500/3811] elapsed=1.5min ETA=10min
  [550/3811] elapsed=1.7min ETA=10min
  [600/3811] elapsed=1.8min ETA=10min
  [650/3811] elapsed=2.0min ETA=10min
  [700/3811] elapsed=2.1min ETA=10min
  [750/3811] elapsed=2.3min ETA=9min
  [800/3811] elapsed=2.4min ETA=9min
  [850/3811] elapsed=2.6min ETA=9min
  [900/3811] elapsed=2.7min ETA=9min
  [950/3811] elapsed=2.9min ETA=9min
  [1000/3811] elapsed=3.0min ETA=8min
  [1050/3811] elapsed=3.2min ETA=8min
  [1100/3811] elapsed=3.3min ETA=8min
  [1150/3811] elapsed=3.5min ETA=8min
  [1200/3811] elapsed=3.6min ETA=8min
  [1250/3811] elapse

In [8]:
# =============================================================================
# Cell 8: Ensure a fallback submission exists (B config, no SAHI)
# =============================================================================
# If SAHI won, this gives you a backup submission in case SAHI submission
# underperforms on the private LB.

if USE_SAHI:
    print("Generating fallback submission (config B, no SAHI)...")
    fallback_rows = []
    for i, (_, row) in enumerate(test_df.iterrows(), start=1):
        img_path = COMP_DIR / row["file_name"]
        if not img_path.exists():
            img_path = Path(row["file_name"])

        res = model.predict(
            source=str(img_path), imgsz=1280, conf=0.10, iou=0.55,
            max_det=500, device=DEVICE, verbose=False,
        )[0]

        if res.boxes is None or len(res.boxes) == 0:
            fallback_rows.append({"image_id": row["image_id"],
                                  "PredictionString": ""})
            continue

        xyxy = res.boxes.xyxy.cpu().numpy()
        cls  = res.boxes.cls.cpu().numpy().astype(int)
        scores = res.boxes.conf.cpu().numpy()
        order = np.argsort(-scores)[:500]
        parts = []
        for idx in order:
            x1, y1, x2, y2 = xyxy[idx]
            w, h = float(x2-x1), float(y2-y1)
            if w <= 0 or h <= 0:
                continue
            parts.append(f"{int(cls[idx])} {float(scores[idx]):.4f} "
                         f"{float(x1):.1f} {float(y1):.1f} {w:.1f} {h:.1f}")
        fallback_rows.append({"image_id": row["image_id"],
                              "PredictionString": " ".join(parts)})

        if i % 100 == 0:
            print(f"  [{i}/{len(test_df)}]")

    fallback_df = pd.DataFrame(fallback_rows)
    fallback_path = OUT_DIR / "submission_fallback.csv"
    fallback_df.to_csv(fallback_path, index=False)
    print(f"✅ Fallback: {fallback_path}")
else:
    print("No SAHI winner — using B config as primary submission.")
    fallback_path = None

No SAHI winner — using B config as primary submission.


In [9]:
# =============================================================================
# Cell 9: Final summary
# =============================================================================
print("\n" + "="*70)
print("PIPELINE COMPLETE")
print("="*70)

print("\nExperiments (300-frame val subset):")
for k, r in results.items():
    print(f"  {k}: mAP50 = {r['mAP50']:.5f}")

print(f"\nWinner: {'SAHI' if USE_SAHI else 'B (no SAHI)'}")

print(f"\nSubmissions ready:")
print(f"  Primary:  {OUT_DIR / 'submission_final.csv'}")
if USE_SAHI and fallback_path is not None:
    print(f"  Fallback: {fallback_path}")

print("\nNext steps:")
print("  1. Download both CSVs from Colab's Files panel")
print("  2. Submit the primary to Kaggle LB")
print("  3. If score < 0.236, also try the fallback")
print("  4. Record both LB scores for the report")

print("\nExpected LB ranges:")
print(f"  Baseline (A): 0.236 (already measured)")
print(f"  Tuned B:      ~0.26–0.30")
print(f"  SAHI C:       ~0.30–0.34")


PIPELINE COMPLETE

Experiments (300-frame val subset):
  A: mAP50 = 0.32418
  B: mAP50 = 0.41208
  C: mAP50 = 0.31721

Winner: B (no SAHI)

Submissions ready:
  Primary:  /kaggle/working/artifacts_sahi/submission_final.csv

Next steps:
  1. Download both CSVs from Colab's Files panel
  2. Submit the primary to Kaggle LB
  3. If score < 0.236, also try the fallback
  4. Record both LB scores for the report

Expected LB ranges:
  Baseline (A): 0.236 (already measured)
  Tuned B:      ~0.26–0.30
  SAHI C:       ~0.30–0.34
